# 02 - Limpeza de dados

Notebook responsável por ler os dados brutos de `data/raw/`, tratar
valores faltantes/inconsistentes, padronizar formatos e datas, e salvar
o resultado limpo em `data/processed/`.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/lavras_clima_2000_2024.csv")
df.shape

## Valores faltando e duplicatas

Antes de qualquer tratamento, checamos se existem valores nulos (`NaN`)
por coluna e se há linhas duplicadas (mesmo dia registrado mais de uma
vez).

In [ ]:
print("Valores faltando por coluna:")
print(df.isna().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())

## Datas: converter e checar dias faltando

A coluna `time` vem como texto (string). Convertendo para datetime
conseguimos usar operações de data e, mais importante, checar se o
calendário está completo (25 anos de dados diários não devem ter
"buracos").

In [ ]:
df["time"] = pd.to_datetime(df["time"])

calendario_completo = pd.date_range(start=df["time"].min(), end=df["time"].max(), freq="D")
dias_faltando = calendario_completo.difference(df["time"])

print("Dias esperados:", len(calendario_completo))
print("Dias faltando:", len(dias_faltando))

## Consistência lógica

Checagens de bom senso: a temperatura máxima do dia nunca deve ser
menor que a mínima, e a precipitação não pode ser negativa.

In [ ]:
max_menor_que_min = (df["temperature_2m_max"] < df["temperature_2m_min"]).sum()
chuva_negativa = (df["precipitation_sum"] < 0).sum()

print("Linhas com max < min:", max_menor_que_min)
print("Linhas com chuva negativa:", chuva_negativa)

## Salvar dado limpo

Nenhum problema encontrado nas checagens acima, então salvamos o
DataFrame (já com a coluna `time` convertida para datetime) em
`data/processed/lavras_clima_limpo.csv`, pronto para a análise
exploratória.

In [ ]:
caminho_saida = "../data/processed/lavras_clima_limpo.csv"
df.to_csv(caminho_saida, index=False)
print(f"Arquivo salvo em {caminho_saida}")